The goal of this notebook is an inheritance analysis which serves to analyze clustering differences to both ABC and HypoMap v1 at the same time.

For that matter, cell type matches will be found between L2 and propagated subclass/C66 labels as well as L3 and propagated supertype/C286 labels.

Matches are defined by L2/L3 cell types comprised of at least 66 % of an ABC/HypoMapv1 label, and will be defined as "inherited" for L2 as these are the ABC subclasses whose names were inherited in the descriptive cell type labels. 

'Inherited and deduplicated' refers to many:1 matches.

If L3 cell types match supertypes/C286 but have not matched their parent subclasses/C66 for L2 this will be handled as no match as the hierarchy is violated. 



In [1]:
import warnings
import os 
from pathlib import Path

from dotenv import dotenv_values
import pandas as pd 
import numpy as np 
import anndata as ad 

warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

OUTDIR = DATADIR / "10_analyses" / "clustering_comparison"
os.makedirs(OUTDIR, exist_ok=True)

OBJECTS_DIR = DATADIR / "9_objects" / "singlecell" 
SC_IN_FP = OBJECTS_DIR / 'hypomap3d_singlecell_control.h5ad'
OBS_EXT_FP = OBJECTS_DIR / 'hypomap3d_singlecell_all_obs_extended.csv'

In [3]:
adata = ad.read_h5ad(SC_IN_FP, backed="r")
obs = adata.obs.loc[:, ["CellID"] + [c for c in adata.obs.columns if "descr" in c] + [c for c in adata.obs.columns if "abc_like" in c]] 
del adata 

obs_ext = pd.read_csv(OBS_EXT_FP, low_memory=False)
obs = obs.merge(obs_ext, on="CellID", how="left")

### Getting a mapper from HypoMap3D's cell types to other atlases cell types and back

We are getting more mappings than strictly required here as these come in handy for other analyses.

In [4]:
from typing import Tuple
import pandas as pd
import anndata as ad

def get_cluster_mapping_summary(
    obs: pd.DataFrame,
    source_cluster_col: str,
    target_cluster_col: str
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Maps clusters from a source clustering to target cluster labels. 
    Computes the most and second most probable target clusters for each source cluster.

    Parameters
    ----------
    obs : pd.DataFrame
        DataFrame object with cell metadata including both clustering columns.

    source_cluster_col : str
        The column in `.obs` representing the source clusters (e.g., subclusters, hierarchical clusters).

    target_cluster_col : str
        The column in `.obs` representing the target clusters (e.g., supertypes, functional types).

    Returns
    -------
    Tuple[pd.DataFrame, pd.DataFrame]
        A tuple with:
        
        1. `long_df`: A DataFrame showing, for each source cluster, the most and second most probable
           target cluster matches, with absolute and relative frequencies, plus residual probability.

        2. `all_probs`: A pivoted DataFrame of normalized mapping probabilities from source to target clusters.

    Example
    -------
    long_df, all_probs = get_cluster_mapping_summary(an_obj, 'subcluster', 'supertype')
    """

    # Step 1: Count occurrences of each source-target pair
    df = (
        obs[[source_cluster_col, target_cluster_col]].dropna() # dropna when mapping between non-propagated clusterings. this will not have an impact for propagated mappings as all cells have values
        .groupby([source_cluster_col, target_cluster_col], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values(by=[source_cluster_col, "count"], ascending=[True, False])
    )

    # Step 2: Extract top matches
    results = []
    for source in df[source_cluster_col].unique():
        subset = df[df[source_cluster_col] == source]
        total = subset["count"].sum()

        if subset.empty:
            # Handle case where source cluster has no target mapping
            results.append({
                "source_label": source,
                "source_clustering": source_cluster_col,
                "target_clustering": target_cluster_col,
                "most_probable_match": pd.NA,
                "most_probable_match_counts_absolute": 0,
                "most_probable_match_counts_relative": pd.NA,
                "second_most_probable_match": pd.NA,
                "second_most_probable_match_counts_absolute": 0,
                "second_most_probable_match_counts_relative": pd.NA,
                "total_num_cells": 0,
                "rest_probabilities": pd.NA
            })
            continue

        top_targets = subset.iloc[:2]  # Get top 2 matches

        most = top_targets.iloc[0]
        second = top_targets.iloc[1] if len(top_targets) > 1 else None

        most_label = most[target_cluster_col]
        most_count = most["count"]
        second_label = second[target_cluster_col] if second is not None else pd.NA
        second_count = second["count"] if second is not None else 0

        results.append({
            "source_label": source,
            "source_clustering": source_cluster_col,
            "target_clustering": target_cluster_col,
            "most_probable_match": most_label,
            "most_probable_match_counts_absolute": most_count,
            "most_probable_match_counts_relative": most_count / total if total > 0 else pd.NA,
            "second_most_probable_match": second_label,
            "second_most_probable_match_counts_absolute": second_count,
            "second_most_probable_match_counts_relative": second_count / total if total > 0 else pd.NA,
            "total_num_cells": total,
            "rest_probabilities": 1 - (most_count + second_count) / total if total > 0 else pd.NA
        })

    long_df = pd.DataFrame(results)

    # Step 3: Create full probability matrix
    all_probs = (
        df.pivot(index=source_cluster_col, columns=target_cluster_col, values="count")
        .div(df.groupby(source_cluster_col)["count"].sum(), axis=0)
        .fillna(0)
    )

    return long_df, all_probs


In [5]:
# map for each of ours to each of theirs in both directions

ours = [f"L{i}_descr" for i in range(1,6)]

theirs = [
    'class_id_label',
    'subclass_id_label',
    'supertype_id_label', 
    'cluster_id_label', 

    'class_id_label_propagated',
    'subclass_id_label_propagated',
    'supertype_id_label_propagated', 
    'cluster_id_label_propagated', 

    'C66_named', 
    'C185_named', 
    'C286_named',
    'C465_named',

    'C66_named_propagated',
    'C185_named_propagated',
    'C286_named_propagated',
    'C465_named_propagated',
    ]

In [6]:

long_dfs = []

# mapping from ours to theirs
for source_clustering in ours:

    for target_clustering in theirs:


        long_df, all_probs = get_cluster_mapping_summary(
            obs = obs,
            source_cluster_col = source_clustering,
            target_cluster_col = target_clustering)
        
        long_dfs.append(long_df)


# mapping from theirs to ours
for source_clustering in theirs:

    for target_clustering in ours:

        long_df, all_probs = get_cluster_mapping_summary(
            obs = obs,
            source_cluster_col = source_clustering,
            target_cluster_col = target_clustering)
        
        long_dfs.append(long_df)

long_df = pd.concat(long_dfs, axis = 0)

In [7]:
long_df.head()

,source_label,source_clustering,target_clustering,most_probable_match,most_probable_match_counts_absolute,most_probable_match_counts_relative,second_most_probable_match,second_most_probable_match_counts_absolute,second_most_probable_match_counts_relative,total_num_cells,rest_probabilities
0,Gb1-2 GABA,L1_descr,class_id_label,12 HY GABA,62872,0.812604,11 CNU-HYa GABA,14499,0.187396,77371,0.0
1,Gl1-1 Glut,L1_descr,class_id_label,14 HY Glut,56698,0.746517,16 HY MM Glut,12398,0.163239,75950,0.090244
2,NN1-1 OPC-Oligo,L1_descr,class_id_label,31 OPC-Oligo,20270,1.0,11 CNU-HYa GABA,0,0.0,20270,0.0
3,NN1-2 Immune,L1_descr,class_id_label,34 Immune,837,1.0,11 CNU-HYa GABA,0,0.0,837,0.0
4,NN1-3 Vascular,L1_descr,class_id_label,33 Vascular,1426,1.0,11 CNU-HYa GABA,0,0.0,1426,0.0


In [8]:
long_df.to_csv(OUTDIR / 'HypoMap3D_to_other_atlas_celltype_mapper.csv', index=False)

# Inheritance analysis

In [9]:
descr_to_abclike_mapper_l2 = obs[["L2_descr", "L2_abc_like"]].drop_duplicates().set_index("L2_descr").to_dict()["L2_abc_like"]
descr_to_abclike_mapper_l3 = obs[["L3_descr", "L3_abc_like"]].drop_duplicates().set_index("L3_descr").to_dict()["L3_abc_like"]
descr_to_abclike_mapper = descr_to_abclike_mapper_l2 | descr_to_abclike_mapper_l3

## ABC

### l2 inheritance

In [10]:
l2 = long_df.loc[
    (long_df.source_clustering == "L2_descr") &
    (long_df.target_clustering == "subclass_id_label_propagated"),
    ["source_label", "most_probable_match", "most_probable_match_counts_relative"]].copy()

l2_n = l2.loc[~l2.source_label.str.contains("NN"),:]

# map to abclike to reuse old code
l2_n.source_label = l2_n.source_label.map(descr_to_abclike_mapper)

l2_n["is_inherited"] = False 
l2_n.loc[l2_n.most_probable_match_counts_relative > 0.66, "is_inherited"] = True

l2_n["hm3d_stem"] = l2_n.source_label.apply(lambda x: " ".join(x.split(" ")[1:]))
l2_n["abc_stem"] = l2_n.most_probable_match.apply(lambda x: " ".join(x.split(" ")[1:]))
l2_n["abc_stem"] = l2_n["abc_stem"].str.replace("Gaba", "GABA", regex=False)


match_counter = l2_n[l2_n.is_inherited].most_probable_match.value_counts().to_dict()
l2_n["num_valid_abc_match_occurrences"] = l2_n.most_probable_match.map(match_counter)
l2_n.loc[~l2_n.is_inherited, "num_valid_abc_match_occurrences"] = np.nan

l2_n["is_deduplicated"] = False 
l2_n.loc[(l2_n.hm3d_stem != l2_n.abc_stem) & (l2_n.num_valid_abc_match_occurrences > 1) & (l2_n.is_inherited), "is_deduplicated"] = True

l2_n[l2_n.is_inherited == False].sort_values(by="num_valid_abc_match_occurrences")

l2_n["inheritance_status"] = "new cluster"
l2_n.loc[l2_n.is_inherited, "inheritance_status"] = "inherited"
l2_n.loc[l2_n.is_deduplicated, "inheritance_status"] = "inherited and deduplicated"

/tmp/ipykernel_3897632/108426248.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  l2_n.source_label = l2_n.source_label.map(descr_to_abclike_mapper)
/tmp/ipykernel_3897632/108426248.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  l2_n["is_inherited"] = False
/tmp/ipykernel_3897632/108426248.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pyd

In [11]:
l2_n.inheritance_status.value_counts(normalize=True)

inheritance_status
inherited                     0.500000
inherited and deduplicated    0.352941
new cluster                   0.147059
Name: proportion, dtype: float64

In [12]:
l2_n.to_csv(OUTDIR / "inheritance_status_l2.csv", index=False)

### l3 inheritance

In [13]:

def classify_mapping(joined_n, match_threshold=0.66):
    """
    Classifies each row into 'new', '1:1', or 'many:1' based on matching strength and suffix of supertype label.
    Modifies the DataFrame in place.
    """

    # Initialize all as 'unknown'
    joined_n['mapping_category'] = 'unknown'


    # Case 1: All non-inherited subclasses are 'new'
    joined_n.loc[~joined_n['is_inherited_subclass'], 'mapping_category'] = 'new'

    # Work on inherited ones
    inherited_mask = joined_n['is_inherited_subclass']
    inherited = joined_n[inherited_mask]

    # Case 2: match score ≤ threshold → 'new'
    low_conf_mask = inherited['most_probable_match_counts_relative_supertype'] <= match_threshold
    match_violates_tree_mask = (inherited.abc_stem_subclass != inherited.abc_stem_subclass_l2) #e.g. N3-142 PVR Six3 Sox3 GABA_12	maps to 0458 AHN Onecut3 Gaba_3 with > 0.66. in those cases the stems from the abc matches will be different from previous level
    combined_mask = low_conf_mask | match_violates_tree_mask
    joined_n.loc[inherited[low_conf_mask].index, 'mapping_category'] = 'new'
    joined_n.loc[inherited[match_violates_tree_mask & ~low_conf_mask ].index, 'mapping_category'] = 'new (violates hierarchy)'

    # Case 3: match score > threshold → check suffix
    high_conf = inherited[~combined_mask].copy()

    # Extract last character
    suffix = high_conf['source_label_supertype'].str[-1]

    # Assign '1:1' where suffix is digit
    is_digit = suffix.str.isdigit()
    joined_n.loc[high_conf[is_digit].index, 'mapping_category'] = '1:1'

    # Assign 'many:1' where suffix is letter
    is_letter = suffix.str.isalpha()
    joined_n.loc[high_conf[is_letter].index, 'mapping_category'] = 'many:1'

    return joined_n


In [14]:
l3 = long_df.loc[
    (long_df.source_clustering == "L3_descr") &
    (long_df.target_clustering == "supertype_id_label_propagated"),
    ["source_label", "most_probable_match", "most_probable_match_counts_relative"]].copy()

l3_n = l3.loc[~l3.source_label.str.contains("NN"),:]

# map to abclike to reuse old code 
l3_n.source_label = l3_n.source_label.map(descr_to_abclike_mapper)

l3_n["hm3d_stem_supertype"] = l3_n.source_label.apply(lambda x: " ".join(x.split(" ")[1:]))
l3_n["abc_stem_supertype"] = l3_n.most_probable_match.apply(lambda x: " ".join(x.split(" ")[1:]))
l3_n["abc_stem_supertype"] = l3_n["abc_stem_supertype"].str.replace("Gaba", "GABA", regex=False)

l3_n["hm3d_stem_subclass"] = l3_n["hm3d_stem_supertype"].apply(lambda x: x.split("_")[0])
l3_n["abc_stem_subclass"] = l3_n["abc_stem_supertype"].apply(lambda x: x.split("_")[0])

l3_n = l3_n.rename(columns = {
    	"source_label": "source_label_supertype",
        "most_probable_match": "most_probable_match_supertype",
        "most_probable_match_counts_relative": "most_probable_match_counts_relative_supertype"
})

l3_n = l3_n.set_index("hm3d_stem_subclass")

l2_n_join = l2_n.copy()
l2_n_join.columns = [c + "_subclass" for c in l2_n_join.columns]
l2_n_join = l2_n_join.set_index("hm3d_stem_subclass")
joined_n = l3_n.join(l2_n_join, how='left', rsuffix='_l2').reset_index()

joined_n = classify_mapping(joined_n)

/tmp/ipykernel_3897632/2579473368.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  l3_n.source_label = l3_n.source_label.map(descr_to_abclike_mapper)
/tmp/ipykernel_3897632/2579473368.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  l3_n["hm3d_stem_supertype"] = l3_n.source_label.apply(lambda x: " ".join(x.split(" ")[1:]))
/tmp/ipykernel_3897632/2579473368.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value

In [15]:
joined_n.mapping_category.value_counts(normalize=True)

mapping_category
1:1                         0.378049
new                         0.361789
many:1                      0.207317
new (violates hierarchy)    0.052846
Name: proportion, dtype: float64

In [16]:
joined_n.to_csv(OUTDIR / "inheritance_status_l3.csv", index=False)

## HMv1 

### l2 to hmv1 "inheritance" 

In [17]:
l2_v1 = long_df.loc[
    (long_df.source_clustering == "L2_descr") &
    (long_df.target_clustering == "C66_named_propagated"),
    ["source_label", "most_probable_match", "most_probable_match_counts_relative"]].copy()

l2_v1_n = l2_v1.loc[~l2_v1.source_label.str.contains("NN"),:]

# map to abclike to reuse old code 
l2_v1_n.source_label = l2_v1_n.source_label.map(descr_to_abclike_mapper)

l2_v1_n["is_inherited"] = False 
l2_v1_n.loc[l2_v1_n.most_probable_match_counts_relative > 0.66, "is_inherited"] = True

match_counter = l2_v1_n[l2_v1_n["is_inherited"]].most_probable_match.value_counts().to_dict()
l2_v1_n["num_valid_hmv1_match_occurrences"] = l2_v1_n.most_probable_match.map(match_counter)
l2_v1_n.loc[~l2_v1_n.is_inherited, "num_valid_hmv1_match_occurrences"] = np.nan

l2_v1_n["hm3d_stem"] = l2_v1_n.source_label.apply(lambda x: " ".join(x.split(" ")[1:]))
l2_v1_n["hmv1_stem"] = l2_v1_n.most_probable_match.apply(lambda x: x.split(" ")[1])
l2_v1_n["hmv1_stem"] = l2_v1_n["hmv1_stem"].str.replace("Gaba", "GABA", regex=False)

# Default to False
l2_v1_n['is_deduplicated'] = False

# Only iterate over rows where is_inherited is True
inherited_rows = l2_v1_n[l2_v1_n['is_inherited']]

# Go through each such row
for idx, row in inherited_rows.iterrows():
    mp_match = row['most_probable_match']
    src_label = row['source_label']
    
    # Subset to all rows with same most_probable_match
    same_match_group = l2_v1_n[l2_v1_n['most_probable_match'] == mp_match]
    
    # Count how many in that group have is_inherited == True
    inherited_count = same_match_group['is_inherited'].sum()
    
    if inherited_count > 1:
        # Set is_deduplicated to True for this specific row
        l2_v1_n.at[idx, 'is_deduplicated'] = True


l2_v1_n["inheritance_status"] = "new cluster"
l2_v1_n.loc[l2_v1_n.is_inherited, "inheritance_status"] = "inherited"
l2_v1_n.loc[l2_v1_n.is_deduplicated, "inheritance_status"] = "inherited and deduplicated"

/tmp/ipykernel_3897632/4203575394.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  l2_v1_n.source_label = l2_v1_n.source_label.map(descr_to_abclike_mapper)
/tmp/ipykernel_3897632/4203575394.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  l2_v1_n["is_inherited"] = False
/tmp/ipykernel_3897632/4203575394.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https:

In [18]:
l2_v1_n.inheritance_status.value_counts(normalize=True)

inheritance_status
inherited and deduplicated    0.411765
inherited                     0.308824
new cluster                   0.279412
Name: proportion, dtype: float64

In [19]:
l2_v1_n.to_csv(OUTDIR / "inheritance_status_l2_hmv1.csv", index=False)

### l3 to hmv1 "inheritance" 

In [20]:
l3_v1 = long_df.loc[
    (long_df.source_clustering == "L3_descr") &
    (long_df.target_clustering == "C286_named_propagated"),
    ["source_label", "most_probable_match", "most_probable_match_counts_relative"]].copy()

l3_v1_n = l3_v1.loc[~l3_v1.source_label.str.contains("NN"),:]

# map to abclike to reuse old code 
l3_v1_n.source_label = l3_v1_n.source_label.map(descr_to_abclike_mapper)

l3_v1_n["hm3d_stem_supertype"] = l3_v1_n.source_label.apply(lambda x: " ".join(x.split(" ")[1:]))
l3_v1_n["hmv1_stem_supertype"] = l3_v1_n.most_probable_match.apply(lambda x: " ".join(x.split(" ")[1:]))

l3_v1_n["hm3d_stem_subclass"] = l3_v1_n["hm3d_stem_supertype"].apply(lambda x: x.split("_")[0])
l3_v1_n["hmv1_stem_subclass"] = l3_v1_n["hmv1_stem_supertype"].apply(lambda x: ".".join(x.split("_")[0].split(".")[-2:]))


l3_v1_n = l3_v1_n.rename(columns = {
    	"source_label": "source_label_supertype",
        "most_probable_match": "most_probable_match_supertype",
        "most_probable_match_counts_relative": "most_probable_match_counts_relative_supertype"
})

l3_v1_n = l3_v1_n.set_index("hm3d_stem_subclass")


l2_v1_n_join = l2_v1_n.copy()
l2_v1_n_join.columns = [c + "_subclass" for c in l2_v1_n_join.columns]
l2_v1_n_join = l2_v1_n_join.set_index("hm3d_stem_subclass")

joined_v1_n = l3_v1_n.join(l2_v1_n_join, how='left', rsuffix='_l2').reset_index()

# Initialize mapping_category
joined_v1_n['mapping_category'] = 'unknown'

# Optional: speed up lookup using a copy of relevant columns
subset_cols = [
    'hm3d_stem_subclass',
    'most_probable_match_supertype',
    'most_probable_match_counts_relative_supertype'
]
lookup_df = joined_v1_n[subset_cols].copy()

# Iterate over rows (using source_label_supertype as identifier)
for idx, row in joined_v1_n.iterrows():
    if not row['is_inherited_subclass']:
        joined_v1_n.at[idx, 'mapping_category'] = 'new' # if the subclass had not been inherited, the supertype would be new
        continue

    if row['most_probable_match_counts_relative_supertype'] <= 0.66: # no match -> new
        joined_v1_n.at[idx, 'mapping_category'] = 'new'
        continue

    if (
        row['most_probable_match_counts_relative_supertype'] > 0.66 and # no match -> new
        row['hmv1_stem_subclass'] != row['hmv1_stem_subclass_l2'] #violates hierarchy of upper level match -> new
    ):
        joined_v1_n.at[idx, 'mapping_category'] = 'new (violates hierarchy)'
        continue

    # Subset the DataFrame for comparison
    # keep the group of supertypes that belong to hm3d subclass, 
    match_mask = (
        (joined_v1_n['hm3d_stem_subclass'] == row['hm3d_stem_subclass']) &
        (joined_v1_n['most_probable_match_supertype'] == row['most_probable_match_supertype'])
    )

    count_strong_matches = ( # count the number of times the most pobable match of the hm3d supertype in question would be inherited
        joined_v1_n.loc[match_mask, 'most_probable_match_counts_relative_supertype'] > 0.66
    ).sum()

    if count_strong_matches == 1:
        joined_v1_n.at[idx, 'mapping_category'] = '1:1'
    elif count_strong_matches > 1:
        joined_v1_n.at[idx, 'mapping_category'] = 'many:1'
    # else: remains 'unknown'


/tmp/ipykernel_3897632/1736020987.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  l3_v1_n.source_label = l3_v1_n.source_label.map(descr_to_abclike_mapper)
/tmp/ipykernel_3897632/1736020987.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  l3_v1_n["hm3d_stem_supertype"] = l3_v1_n.source_label.apply(lambda x: " ".join(x.split(" ")[1:]))
/tmp/ipykernel_3897632/1736020987.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_inde

In [21]:
joined_v1_n.mapping_category.value_counts(normalize=True)

mapping_category
new                         0.504065
1:1                         0.231707
many:1                      0.227642
new (violates hierarchy)    0.036585
Name: proportion, dtype: float64

In [22]:
joined_v1_n.to_csv(OUTDIR / "inheritance_status_l3_hmv1.csv", index=False)